# Finite and boundary elements on a sphere

A mesh of the unit ball from `pytta.mesh.distmesh` (Persson and Strang) feeds both solvers:

1. `pytta.fem` gives the eigenfrequencies of the rigid sphere, compared with the roots of j_l'(ka) = 0;
2. `pytta.bem`, on the boundary of the same mesh, gives the field of a pulsating sphere, compared with the analytic one.

In [1]:
import numpy as np
import pytta
from pytta import mesh, fem, bem

nodes, tets = mesh.distmesh(mesh.sphere([0, 0, 0], 1.), 0.15, [[-1, -1, -1], [1, 1, 1]])
q = mesh.quality(nodes, tets)
print(f'{len(nodes)} nodes, {len(tets)} tetrahedra, quality min {q.min():.3f}, mean {q.mean():.3f}')

1229 nodes, 6029 tetrahedra, quality min 0.039, mean 0.894


The first nonzero modes of a rigid sphere: ka = 2.0816 (three modes, l = 1), 3.3421 (five, l = 2) and 4.4934 (one, l = 0).

In [2]:
c = 343.
f, _ = fem.modes(nodes, tets, 10, c=c)
ka = 2 * np.pi * f[1:] / c
exact = np.r_[[2.0816] * 3, [3.3421] * 5, 4.4934]
for k, e in zip(ka, exact):
    print(f'ka {k:.4f}   exact {e:.4f}   error {100 * (k / e - 1):+.2f} %')

ka 2.0953   exact 2.0816   error +0.66 %
ka 2.0957   exact 2.0816   error +0.68 %
ka 2.0963   exact 2.0816   error +0.70 %
ka 3.3843   exact 3.3421   error +1.26 %
ka 3.3847   exact 3.3421   error +1.27 %
ka 3.3862   exact 3.3421   error +1.32 %
ka 3.3863   exact 3.3421   error +1.32 %
ka 3.3876   exact 3.3421   error +1.36 %
ka 4.5979   exact 4.4934   error +2.33 %


The pulsating sphere of radius 0.1 m at ka = 1, on the triangles of the boundary of the volume mesh.

In [3]:
a, rho = 0.1, 1.21
surface = mesh.boundary(nodes, tets)
k = 1. / a
freq = k * c / (2 * np.pi)
p_surface = bem.surface_pressure(a * nodes, surface, freq, 1., c=c, rho=rho)
points = np.array([[0, 0, 1.], [0.5, 0.2, 0.1], [-0.3, -0.3, 0.6]])
r = np.linalg.norm(points, axis=1)
p_field = bem.field(a * nodes, surface, freq, p_surface, 1., points, c=c, rho=rho)
exact = rho * c * 1j * k * a / (1 + 1j * k * a) * a / r * np.exp(-1j * k * (r - a))
for ri, pf, pe in zip(r, p_field, exact):
    print(f'r = {ri:.3f} m: |p| {abs(pf):.4f} Pa, exact {abs(pe):.4f} Pa, error {abs(pf / pe - 1):.1e}')

r = 1.000 m: |p| 29.1942 Pa, exact 29.3471 Pa, error 5.7e-03
r = 0.548 m: |p| 53.3018 Pa, exact 53.5801 Pa, error 5.7e-03
r = 0.735 m: |p| 39.7278 Pa, exact 39.9363 Pa, error 5.7e-03
